In [2]:
import pandas as pd
import sqlite3
from pathlib import Path

# Path ko automatically project root par set karo
BASE_DIR = Path.cwd()
if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent

cleaned_path = BASE_DIR / "data" / "cleaned"

# 1. Cleaned data load karo exact path se
posts_df = pd.read_csv(cleaned_path / "Social_Engine_Posts_Cleaned.csv")
users_df = pd.read_csv(cleaned_path / "Social_Engine_Users_Cleaned.csv")

# 2. SQLite database connection banao
# Ye file tumhare folder me 'social_engine.db' naam se save ho jayegi
conn = sqlite3.connect("social_engine.db")

# 3. Dataframes ko SQL tables me convert karo
users_df.to_sql("users", conn, if_exists="replace", index=False)
posts_df.to_sql("posts", conn, if_exists="replace", index=False)

print("SQL Database 'social_engine.db' ready ho gaya hai!")
print("Tables created: 'users' aur 'posts'")

SQL Database 'social_engine.db' ready ho gaya hai!
Tables created: 'users' aur 'posts'


In [3]:
# Challenge 1: Anomaly Discovery in Post Volume (Window Functions)
query_1 = """
WITH MonthlyStats AS (
    SELECT 
        STRFTIME('%Y-%m', cleaned_timestamp) AS month_year,
        COUNT(post_id) AS total_posts,
        SUM(total_engagement) AS total_monthly_engagement
    FROM posts
    GROUP BY STRFTIME('%Y-%m', cleaned_timestamp)
)
SELECT 
    month_year,
    total_posts,
    total_monthly_engagement,
    LAG(total_posts) OVER (ORDER BY month_year) AS prev_month_posts,
    ROUND((CAST(total_posts AS FLOAT) - LAG(total_posts) OVER (ORDER BY month_year)) / 
          LAG(total_posts) OVER (ORDER BY month_year) * 100, 2) AS mom_growth_percentage
FROM MonthlyStats;
"""

print("--- CHALLENGE 1: SYSTEM FAILURE ANOMALY DETECTION ---")
result_1 = pd.read_sql_query(query_1, conn)
display(result_1)

--- CHALLENGE 1: SYSTEM FAILURE ANOMALY DETECTION ---


,month_year,total_posts,total_monthly_engagement,prev_month_posts,mom_growth_percentage
0,2024-01,110,451146.0,NaN,NaN
1,2024-02,110,418269.0,110.0,0.00
2,2024-03,87,341835.0,110.0,-20.91
3,2024-04,110,456765.0,87.0,26.44
4,2024-05,975,3893472.0,110.0,786.36
5,2024-06,923,3699319.0,975.0,-5.33
6,2024-07,953,3804890.0,923.0,3.25
7,2024-08,974,3963459.0,953.0,2.20
8,2024-09,954,3844783.0,974.0,-2.05
9,2024-10,965,3882395.0,954.0,1.15


In [4]:
# Challenge 2: User Behavioural Grouping (CTEs & Aggregation)
query_2 = """
WITH UserEngagement AS (
    SELECT 
        u.user_id,
        u.location,
        u.follower_count,
        COUNT(p.post_id) AS total_posts,
        SUM(p.total_engagement) AS lifetime_engagement
    FROM users u
    LEFT JOIN posts p ON u.user_id = p.user_id
    GROUP BY u.user_id, u.location, u.follower_count
)
SELECT 
    CASE 
        WHEN lifetime_engagement >= 30000 THEN 'High Impact (Tier 1)'
        WHEN lifetime_engagement BETWEEN 10000 AND 29999 THEN 'Medium Impact (Tier 2)'
        ELSE 'Low Impact (Tier 3)'
    END AS engagement_tier,
    COUNT(user_id) AS user_count,
    ROUND(AVG(total_posts), 2) AS avg_posts_per_user,
    ROUND(AVG(follower_count), 2) AS avg_follower_count
FROM UserEngagement
GROUP BY engagement_tier
ORDER BY user_count DESC;
"""

print("\n--- CHALLENGE 2: BEHAVIOURAL GROUPING & ENGAGEMENT TIERS ---")
result_2 = pd.read_sql_query(query_2, conn)
display(result_2)


--- CHALLENGE 2: BEHAVIOURAL GROUPING & ENGAGEMENT TIERS ---


,engagement_tier,user_count,avg_posts_per_user,avg_follower_count
0,High Impact (Tier 1),807,9.96,25039.12
1,Medium Impact (Tier 2),661,5.87,24810.20
2,Low Impact (Tier 3),32,2.50,26247.56


In [1]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("social_engine.db")

# Easy: E4
query_e4 = """
SELECT post_id, platform, likes, shares, comments
FROM posts
WHERE shares > 1500 AND likes < 500
ORDER BY shares DESC
LIMIT 5;
"""

# Medium: M2
query_m2 = """
WITH UserEngagement AS (
    SELECT 
        u.user_id,
        CASE WHEN u.follower_count >= 25000 THEN 'High follower (>= 25k)' ELSE 'Low follower (< 25k)' END AS follower_group,
        p.total_engagement
    FROM users u
    JOIN posts p ON u.user_id = p.user_id
)
SELECT 
    follower_group,
    COUNT(user_id) AS total_posts,
    ROUND(AVG(total_engagement), 2) AS avg_engagement_per_post
FROM UserEngagement
GROUP BY follower_group;
"""

# Hard: H6
query_h6 = """
WITH OverallAvg AS (
    SELECT AVG(total_engagement) AS global_avg FROM posts
),
UserStats AS (
    SELECT 
        u.user_id,
        u.location,
        u.follower_count,
        COUNT(p.post_id) AS num_posts,
        SUM(p.total_engagement) AS total_engagement,
        AVG(p.total_engagement) AS avg_engagement,
        SUM(CASE WHEN p.shares > p.likes THEN 1 ELSE 0 END) AS suspicious_posts_count
    FROM users u
    JOIN posts p ON u.user_id = p.user_id
    GROUP BY u.user_id, u.location, u.follower_count
)
SELECT 
    us.user_id,
    us.location,
    us.follower_count,
    us.num_posts,
    ROUND(us.avg_engagement, 2) AS avg_engagement,
    us.total_engagement
FROM UserStats us
CROSS JOIN OverallAvg oa
WHERE us.follower_count < 10000
  AND us.avg_engagement > oa.global_avg
  AND us.suspicious_posts_count > 0
ORDER BY us.total_engagement DESC;
"""

print("--- E4: Highly Shared but Poorly Liked ---")
display(pd.read_sql_query(query_e4, conn))

print("\n--- M2: Follower Group Engagement ---")
display(pd.read_sql_query(query_m2, conn))

print("\n--- H6: Suspicious High Impact Users ---")
display(pd.read_sql_query(query_h6, conn))

--- E4: Highly Shared but Poorly Liked ---


,post_id,platform,likes,shares,comments
0,euvr0r10wrj6,Facebook,453.0,2000,408
1,oiszojqm6qnn,Instagram,390.0,1999,858
2,f2e5kdfldedz,Unknown,447.0,1997,641
3,mgv7p46wzpek,Reddit,252.0,1993,971
4,u1aa801qvxeu,Unknown,162.0,1992,306



--- M2: Follower Group Engagement ---


,follower_group,total_posts,avg_engagement_per_post
0,High follower (>= 25k),5925,4004.97
1,Low follower (< 25k),6075,4008.78



--- H6: Suspicious High Impact Users ---


,user_id,location,follower_count,num_posts,avg_engagement,total_engagement
0,user_uerv85na,"Rome, Italy",1824,16,4522.56,72361.0
1,user_68enpikx,"Houston, USA",9364,15,4526.20,67893.0
2,user_fgjkkrie,"Lyon, France",2211,14,4477.86,62690.0
3,user_9mtets0p,"Johannesburg, South Africa",8262,14,4458.14,62414.0
4,user_cuzig14g,"Cairo, Egypt",8070,15,4007.73,60116.0
...,...,...,...,...,...,...
87,user_h8g2fyv0,"London, UK",4209,5,4135.00,20675.0
88,user_pp3dgfak,"Seoul, South Korea",109,5,4015.60,20078.0
89,user_5td0e1us,Singapore,6747,3,4754.33,14263.0
90,user_rscyqide,"Shanghai, China",4478,3,4514.33,13543.0
